# Next Steps: Multi-Class, Custom Channels, and Production Inference

> **Prerequisite:** complete `01_training_walkthrough.ipynb` first.
>
> **What you will learn:** extend the same library to harder tasks — three-class masks, a custom 4-channel input, a larger encoder, parallel dataset build, test-time augmentation, and high-recall threshold tuning.

Every section is self-contained with synthetic data so it runs on the Linux HPC with no external files.

| Section | Topic |
|---------|-------|
| 1 | Multi-class segmentation (3 classes) |
| 2 | Custom channel ops (4-channel input) |
| 3 | Larger encoder + parallel patch build |
| 4 | Test-time augmentation (TTA) |
| 5 | High-recall threshold tuning |
| 6 | Per-class metrics and multi-class export |

## Setup

In [ ]:
from pathlib import Path
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from segmentation_tools_pt.core          import set_seed, get_device, SegConfig
from segmentation_tools_pt.channels      import (
    CHANNEL_OPS, build_image, stack_channels, linear_diff, product, abs_diff, tophat,
)
from segmentation_tools_pt.masks         import fit_rotated_rects_multiclass, one_hot
from segmentation_tools_pt.patches       import PatchConfig
from segmentation_tools_pt.dataset_build import build_patch_dataset
from segmentation_tools_pt.transforms    import compute_dataset_stats
from segmentation_tools_pt.dataset       import build_datasets, build_dataloaders
from segmentation_tools_pt.losses        import SegLoss
from segmentation_tools_pt.metrics       import SegMetrics, ObjectMetrics
from segmentation_tools_pt.model         import get_model, get_param_groups
from segmentation_tools_pt.train         import train_one_epoch, validate
from segmentation_tools_pt.inference     import predict, decode
from segmentation_tools_pt.visualize     import sanity_check, show_pred
from segmentation_tools_pt.export        import export_onnx

set_seed(42)
DEVICE = get_device()
WORK_DIR = Path('tutorial_next_steps')
WORK_DIR.mkdir(exist_ok=True)
print('Device:', DEVICE)

---
## 1 — Multi-class segmentation

Binary segmentation is `num_classes == 1`. For **multi-class**, masks are integer-labelled: `0` = background, `1..K` = foreground classes.

The same modules adapt automatically:
- `get_model(..., num_classes=K)` → softmax logits `(B, K, H, W)`
- `SegLoss` / `SegMetrics` → per-class then mean
- `decode(probs)` → `argmax` over classes

We synthesise a 3-class dataset: class 1 = small squares, class 2 = horizontal bars, class 3 = vertical bars.

In [ ]:
NUM_CLASSES = 3
CFG_MC = SegConfig(in_channels=3, num_classes=NUM_CLASSES)
print('Multi-class task:', CFG_MC, '| is_binary:', CFG_MC.is_binary)

RAW_MC = WORK_DIR / 'raw_multiclass'
(RAW_MC / 'images').mkdir(parents=True, exist_ok=True)
(RAW_MC / 'masks').mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(1)
H, W = 200, 200

for i in range(6):
    sid = f'mc_{i:02d}'
    ch0 = rng.integers(50, 100, (H, W), np.uint8)
    ch1 = rng.integers(40, 90, (H, W), np.uint8)
    mask = np.zeros((H, W), np.uint8)

    # class 1: square
    y, x = int(rng.integers(10, 60)), int(rng.integers(10, 60))
    mask[y:y+30, x:x+30] = 1
    ch0[y:y+30, x:x+30] = 200; ch1[y:y+30, x:x+30] = 220

    # class 2: horizontal bar
    y = int(rng.integers(80, 120))
    mask[y:y+12, 20:120] = 2
    ch0[y:y+12, 20:120] = 180; ch1[y:y+12, 20:120] = 200

    # class 3: vertical bar
    x = int(rng.integers(130, 170))
    mask[30:130, x:x+12] = 3
    ch0[30:130, x:x+12] = 170; ch1[30:130, x:x+12] = 190

    cv2.imwrite(str(RAW_MC / 'images' / f'{sid}_ch0.png'), ch0)
    cv2.imwrite(str(RAW_MC / 'images' / f'{sid}_ch1.png'), ch1)
    cv2.imwrite(str(RAW_MC / 'masks'  / f'{sid}_mask.png'), mask)

print(f'Created {6} multi-class samples in {RAW_MC}')

In [ ]:
# Inspect one mask: three distinct integer labels
m = cv2.imread(str(RAW_MC / 'masks' / 'mc_00_mask.png'), cv2.IMREAD_GRAYSCALE)
print('Unique labels:', np.unique(m).tolist())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
ch0 = cv2.imread(str(RAW_MC / 'images' / 'mc_00_ch0.png'), cv2.IMREAD_GRAYSCALE)
ch1 = cv2.imread(str(RAW_MC / 'images' / 'mc_00_ch1.png'), cv2.IMREAD_GRAYSCALE)
axes[0].imshow(ch0, cmap='gray'); axes[0].set_title('CH0'); axes[0].axis('off')
axes[1].imshow(ch1, cmap='gray'); axes[1].set_title('CH1'); axes[1].axis('off')
axes[2].imshow(m, cmap='tab10', vmin=0, vmax=3); axes[2].set_title('Integer mask'); axes[2].axis('off')
oh = one_hot(m, NUM_CLASSES)
axes[3].imshow(oh.argmax(0), cmap='tab10', vmin=0, vmax=3); axes[3].set_title('one_hot → argmax'); axes[3].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Build patches — note num_classes=3 and integer masks saved as-is
PATCH_MC = WORK_DIR / 'patches_multiclass'
patch_cfg = PatchConfig(patch_size=80, stride=40, min_fg_pixels=50,
                        max_empty_fraction=0.05, val_fraction=0.33, seed=42)

meta_mc = build_patch_dataset(
    source_dir=str(RAW_MC), output_dir=str(PATCH_MC),
    cfg=patch_cfg, num_classes=NUM_CLASSES, n_channels=2, n_jobs=1,
)
print(f"Train: {len(meta_mc['train'])} patches | Val: {len(meta_mc['val'])} patches")

In [ ]:
# Train a small multi-class model (short run for the tutorial)
CHANNEL_NAMES = ['ch0', 'ch1', 'product']

def _iter_images(patch_dir, split='train'):
    img_dir = Path(patch_dir) / split / 'images'
    for m in json.loads((Path(patch_dir) / 'metadata.json').read_text())[split]:
        sid = m['sample_id']
        chans = [cv2.imread(str(img_dir / f'{sid}_ch{i}.png'), cv2.IMREAD_GRAYSCALE) for i in range(2)]
        if all(c is not None for c in chans):
            yield build_image(chans, CHANNEL_NAMES)

MEAN, STD = compute_dataset_stats(_iter_images(PATCH_MC))

train_loader, val_loader, train_ds, val_ds = build_dataloaders(
    root=str(PATCH_MC), batch_size=4, num_workers=0, crop_size=48,
    num_classes=NUM_CLASSES, n_source_channels=2, channel_names=CHANNEL_NAMES,
    mean=MEAN, std=STD,
)

model_mc = get_model('mobilenet_v2', in_channels=3, num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
criterion = SegLoss().to(DEVICE)
optimizer = AdamW(model_mc.parameters(), lr=1e-3)

for epoch in range(5):
    criterion.set_phase(1)
    tr = train_one_epoch(model_mc, train_loader, criterion, optimizer, DEVICE, num_classes=NUM_CLASSES)
    va = validate(model_mc, val_loader, criterion, DEVICE, num_classes=NUM_CLASSES)
    print(f'Ep {epoch} | loss {tr["loss"]:.3f} | val_iou {va["iou"]:.3f} | val_dice {va["dice"]:.3f}')

print('\nPer-class IoU:')
for cls, scores in va['per_class'].items():
    print(f'  class {cls}: IoU={scores["iou"]:.3f}  recall={scores["recall"]:.3f}')

In [ ]:
# Multi-class decode: argmax over softmax probabilities
sample = val_ds[0]
with torch.no_grad():
    probs = predict(model_mc, sample['image'], DEVICE)   # (3, H, W)
pred_labels = decode(probs)                             # (H, W) integer labels

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(sample['image'][0].cpu(), cmap='gray'); axes[0].set_title('Input ch0'); axes[0].axis('off')
axes[1].imshow(sample['mask'].cpu(), cmap='tab10', vmin=0, vmax=3); axes[1].set_title('Ground truth'); axes[1].axis('off')
axes[2].imshow(pred_labels.cpu(), cmap='tab10', vmin=0, vmax=3); axes[2].set_title('Prediction (argmax)'); axes[2].axis('off')
plt.tight_layout(); plt.show()
print('Predicted labels:', torch.unique(pred_labels).tolist())

---
## 2 — Custom channel ops (4-channel input)

The default input is 3 channels: `[ch0, ch1, product]`. You can extend it without changing library code — register a new op and pass a longer `channel_names` list.

Here we add `tophat_plus_product`: the mean of top-hat and product, which highlights small bright structures that are also bright in both captures.

In [ ]:
# Register a custom op (do this once at the top of your training script or notebook)
def tophat_plus_product(imgs):
    th = tophat(imgs[1], k=15)
    pr = product(imgs[0], imgs[1])
    return ((th.astype(np.uint16) + pr.astype(np.uint16)) // 2).astype(np.uint8)

CHANNEL_OPS['tophat_plus_product'] = tophat_plus_product

# 4-channel recipe: raw captures + abs_diff + tophat_plus_product
CUSTOM_CHANNELS = ['ch0', 'ch1', 'abs_diff', 'tophat_plus_product']
print('Custom channel recipe:', CUSTOM_CHANNELS)
print('Model in_channels must be:', len(CUSTOM_CHANNELS))

In [ ]:
# Visualise all four channels on one sample
ch0 = cv2.imread(str(RAW_MC / 'images' / 'mc_00_ch0.png'), cv2.IMREAD_GRAYSCALE)
ch1 = cv2.imread(str(RAW_MC / 'images' / 'mc_00_ch1.png'), cv2.IMREAD_GRAYSCALE)
img4 = build_image([ch0, ch1], CUSTOM_CHANNELS)   # (H, W, 4)

fig, axes = plt.subplots(1, len(CUSTOM_CHANNELS), figsize=(4 * len(CUSTOM_CHANNELS), 4))
for i, name in enumerate(CUSTOM_CHANNELS):
    axes[i].imshow(img4[..., i], cmap='gray')
    axes[i].set_title(name); axes[i].axis('off')
plt.suptitle('4-channel model input')
plt.tight_layout(); plt.show()
print('Shape:', img4.shape)

In [ ]:
# Build a model with 4 input channels — only in_channels and channel_names change
CFG_4CH = SegConfig(in_channels=4, num_classes=1)
model_4ch = get_model('mobilenet_v2', in_channels=4, num_classes=1, pretrained=False)
dummy = torch.randn(1, 4, 64, 64)
print('4-channel model output:', tuple(model_4ch(dummy).shape))

# When building dataloaders, pass the same channel_names:
# build_dataloaders(..., channel_names=CUSTOM_CHANNELS, mean=(0,0,0,0), std=(1,1,1,1))

---
## 3 — Larger encoder and parallel dataset build

**Larger encoder:** swap `'mobilenet_v2'` for `'resnet34'` or `'efficientnet-b0'`. The API is identical; only memory and speed change.

**Parallel build:** pass `n_jobs=8` to `build_patch_dataset` to fan out patch extraction across CPU cores on the HPC.

In [ ]:
# Compare encoder sizes (pretrained=False for speed in the tutorial)
for arch in ['mobilenet_v2', 'resnet34', 'efficientnet-b0']:
    m = get_model(arch, in_channels=3, num_classes=1, pretrained=False)
    total = sum(p.numel() for p in m.parameters()) / 1e6
    enc   = sum(p.numel() for p in m.encoder.parameters()) / 1e6
    print(f'{arch:<20} total={total:.2f}M  encoder={enc:.2f}M')

In [ ]:
# Parallel dataset build — set n_jobs to the number of CPU cores on your HPC node
# (use n_jobs=1 in this notebook if you want a deterministic quick test)
import time

PATCH_PARALLEL = WORK_DIR / 'patches_parallel'
t0 = time.perf_counter()
meta_par = build_patch_dataset(
    source_dir=str(RAW_MC), output_dir=str(PATCH_PARALLEL),
    cfg=patch_cfg, num_classes=NUM_CLASSES, n_channels=2,
    n_jobs=2,   # <-- increase to 8 or 16 on the HPC
)
elapsed = time.perf_counter() - t0
print(f'Built {len(meta_par["train"]) + len(meta_par["val"])} patches in {elapsed:.1f}s with n_jobs=2')

---
## 4 — Test-time augmentation (TTA)

At inference, average predictions over 90° rotations. This usually improves recall on thin structures at the cost of 4× compute. Use it on validation patches where pixel IoU matters most.

In [ ]:
sample = val_ds[0]
model_mc.eval()

probs_no_tta = predict(model_mc, sample['image'], DEVICE, tta=False)
probs_tta    = predict(model_mc, sample['image'], DEVICE, tta=True, n_augs=4)

pred_no_tta = decode(probs_no_tta)
pred_tta    = decode(probs_tta)
gt          = sample['mask']

# Pixel IoU for this single patch
def patch_iou(pred, gt):
    p, t = pred.bool(), gt.bool()
    return float((p & t).sum()) / float((p | t).sum() + 1e-6)

iou_no_tta = patch_iou(pred_no_tta > 0, gt > 0)
iou_tta    = patch_iou(pred_tta > 0, gt > 0)
print(f'Foreground IoU without TTA: {iou_no_tta:.4f}')
print(f'Foreground IoU with    TTA: {iou_tta:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(gt.cpu(), cmap='tab10', vmin=0, vmax=3); axes[0].set_title('GT'); axes[0].axis('off')
axes[1].imshow(pred_no_tta.cpu(), cmap='tab10', vmin=0, vmax=3); axes[1].set_title(f'No TTA (IoU={iou_no_tta:.3f})'); axes[1].axis('off')
axes[2].imshow(pred_tta.cpu(),    cmap='tab10', vmin=0, vmax=3); axes[2].set_title(f'TTA (IoU={iou_tta:.3f})'); axes[2].axis('off')
plt.tight_layout(); plt.show()

---
## 5 — High-recall threshold tuning (binary)

For inspection tasks, missing a target is worse than a false alarm. `find_operating_threshold` finds the **highest** sigmoid threshold that still meets your recall target.

We collapse the multi-class masks to binary (any label → foreground) and rebuild patches so `num_classes=1` loads correctly.

In [ ]:
from segmentation_tools_pt.inference import find_operating_threshold

# Binary task needs masks saved as 0/255 (not integer labels 1,2,3).
# Collapse multi-class raw masks to binary foreground and rebuild patches.
RAW_BIN = WORK_DIR / 'raw_binary'
(RAW_BIN / 'images').mkdir(parents=True, exist_ok=True)
(RAW_BIN / 'masks').mkdir(parents=True, exist_ok=True)
for p in (RAW_MC / 'images').glob('*_ch0.png'):
    sid = p.name.replace('_ch0.png', '')
    ch0 = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    ch1 = cv2.imread(str(RAW_MC / 'images' / f'{sid}_ch1.png'), cv2.IMREAD_GRAYSCALE)
    mc  = cv2.imread(str(RAW_MC / 'masks'  / f'{sid}_mask.png'), cv2.IMREAD_GRAYSCALE)
    cv2.imwrite(str(RAW_BIN / 'images' / f'{sid}_ch0.png'), ch0)
    cv2.imwrite(str(RAW_BIN / 'images' / f'{sid}_ch1.png'), ch1)
    cv2.imwrite(str(RAW_BIN / 'masks'  / f'{sid}_mask.png'), ((mc > 0) * 255).astype(np.uint8))

PATCH_BIN = WORK_DIR / 'patches_binary'
build_patch_dataset(str(RAW_BIN), str(PATCH_BIN), cfg=patch_cfg, num_classes=1, n_channels=2, n_jobs=1)

train_loader_bin, val_loader_bin, _, val_ds_bin = build_dataloaders(
    root=str(PATCH_BIN), batch_size=4, num_workers=0, crop_size=48,
    num_classes=1, n_source_channels=2, channel_names=CHANNEL_NAMES,
    mean=MEAN, std=STD,
)

model_bin = get_model('mobilenet_v2', in_channels=3, num_classes=1, pretrained=False).to(DEVICE)
opt_bin = AdamW(model_bin.parameters(), lr=1e-3)
crit_bin = SegLoss().to(DEVICE)
for _ in range(5):
    train_one_epoch(model_bin, train_loader_bin, crit_bin, opt_bin, DEVICE, num_classes=1)

# Sweep thresholds — aim for very high recall (e.g. 99.7% in production)
for target in [0.90, 0.95, 0.99]:
    res = find_operating_threshold(model_bin, val_loader_bin, DEVICE, target_recall=target)
    print(f"target_recall={target:.2f} → threshold={res['recommended_threshold']}  "
          f"recall={res['achieved_recall']:.4f}  precision={res['achieved_precision']:.4f}")

---
## 6 — Per-class object metrics and ONNX export

Pixel IoU can look good while whole objects are missed. Run `ObjectMetrics` **per class** by masking each label separately.

Export the multi-class model with `num_classes=K` in the ONNX trace — the output is `(B, K, H, W)` logits.

In [ ]:
# Per-class object detection on the validation set
model_mc.eval()
class_det = {c: ObjectMetrics(overlap_thresh=0.5) for c in range(1, NUM_CLASSES + 1)}

with torch.no_grad():
    for batch in val_loader:
        for i in range(batch['image'].shape[0]):
            probs = predict(model_mc, batch['image'][i], DEVICE)
            pred  = decode(probs).cpu().numpy()
            gt    = batch['mask'][i].numpy()
            for cls in range(1, NUM_CLASSES + 1):
                class_det[cls].update((pred == cls).astype(np.uint8), (gt == cls).astype(np.uint8))

print('Per-class object detection (one val batch):')
for cls, acc in class_det.items():
    d = acc.compute()
    print(f'  class {cls}: detected {d["detected"]}/{d["total_objects"]}  '
          f'det_rate={d["detection_rate"]:.3f}  false_alarms={d["false_alarms"]}')

In [ ]:
# Export multi-class model — output shape (B, K, H, W)
ONNX_MC = str(WORK_DIR / 'model_multiclass.onnx')
export_onnx(model_mc, ONNX_MC, in_channels=3, input_size=48, verify=True)
print('Exported:', ONNX_MC)

---
## Summary

| What changed | Binary (tutorial 1) | This notebook |
|---|---|---|
| Mask format | `0 / 255` | integer labels `0, 1, 2, 3` |
| `num_classes` | `1` | `3` |
| Model output | sigmoid `(B,1,H,W)` | softmax `(B,3,H,W)` |
| Decode | `threshold` | `argmax` |
| Input channels | 3 (`ch0,ch1,product`) | 4 possible via `CUSTOM_CHANNELS` |
| Inference | single forward | optional TTA |
| Threshold | fixed 0.5 | recall-driven sweep |

### Checklist before a real production run

1. Replace synthetic data with your `{id}_ch{i}.png` / `{id}_mask.png` layout.
2. Run `compute_dataset_stats` on the **full** training set.
3. Set `n_jobs` to your HPC core count for `build_patch_dataset`.
4. Train 60+ epochs with `pretrained=True` and a larger encoder.
5. Run `find_operating_threshold` on a held-out validation set.
6. Export ONNX and verify with `onnxruntime` on the deployment machine.
7. Enable TTA only if latency budget allows (4× slower).